# Indexación del catálogo SQLite en ChromaDB

Este notebook prepara documentos y embeddings, verifica un respaldo, retira la colección antigua y carga `catalogo_cve_2023_2026`. SQLite se consulta en modo de solo lectura. No necesita la API de NVD ni el `.env`.

## 1. Configuración y diagnóstico

In [1]:
from pathlib import Path
import sys, json, collections
import chromadb
ROOT = Path.cwd()
if not (ROOT / 'preprosesamiento').exists():
    ROOT = ROOT.parent
PROCESO = ROOT / 'preprosesamiento'
sys.path.insert(0, str(PROCESO / 'chroma'))
import indexacion as ix
DB = PROCESO / 'sqlite' / 'catalogo.sqlite3'
SALIDAS = PROCESO / 'chroma' / 'salidas'
HOST, PORT = 'localhost', 8001
COLECCION_ANTERIOR = ix.OLD_COLLECTION
COLECCION = ix.COLLECTION
client = chromadb.HttpClient(host=HOST, port=PORT)
client.heartbeat()
rows, cpes = ix.catalogue(DB)
print('Colecciones:', sorted(ix.names(client)))
print('CVE SQLite:', len(rows), 'CPE:', len(cpes))
print('CVE por año:', dict(collections.Counter(r['published'][:4] for r in rows)))

Colecciones: ['catalogo_cve_2023_2026']
CVE SQLite: 1000 CPE: 9081
CVE por año: {'2026': 250, '2023': 250, '2024': 250, '2025': 250}


## 2. Modelo y preparación de documentos

Se utiliza ONNX MiniLM local con 384 dimensiones. El tokenizador sin truncamiento mide el documento completo, incluyendo encabezados y tokens especiales: máximo 240 tokens, solapamiento de hasta 30. Se conserva el texto original; consultas de ejemplo en inglés. CWE sin contexto no genera texto inventado. CAPEC se deduplica dentro de cada CVE.

In [2]:
tokenizer, model, model_hash = ix.tokenizer_and_model()
records, manifest, run = ix.prepare(DB, SALIDAS, tokenizer, model_hash)
print('Carga:', manifest['run_id'])
print('Documentos:', manifest['document_count'])
print('Fragmentos por tipo:', manifest['chunks_by_type'])
print('Cobertura:', manifest['coverage'])
print('Asociaciones CWE sin contexto:', len(manifest['missing_cwe']))
print('Ejemplo:', records[0]['document'])

Carga: 04ee8166f572dec8c2f6f9e78ea3042c4786512e20cbe2f9dff2c38a3e8dd4ab
Documentos: 68973
Fragmentos por tipo: {'nvd': 1054, 'cwe': 5675, 'capec': 53156, 'cpe': 9081, 'exploit': 7}
Cobertura: {'nvd': 1000, 'cve_with_cwe': 971, 'cwe': 2232, 'cve_with_capec': 802, 'capec': 44168, 'cpe': 9081, 'cve_with_exploit': 6, 'exploit': 7}
Asociaciones CWE sin contexto: 500
Ejemplo: CVE-2017-20253 | nvd | CVE-2017-20253 | description_0
Joomla! Component My Projects 2.0 contains an SQL injection vulnerability that allows unauthenticated attackers to execute arbitrary SQL queries by injecting malicious code through the VerAyari parameter. Attackers can craft requests to the component endpoint with SQL injection payloads to extract sensitive database information including credentials and system data.


## 3. Embeddings locales y caché reanudable

Esta etapa finaliza y verifica todos los vectores antes de eliminar datos. Una interrupción continúa desde el último lote persistido.

In [3]:
vectors = ix.embeddings(records, run, model)
print('Vectores verificados:', vectors.shape)
print('Hash:', json.loads((run / 'embeddings_verificados.json').read_text())['sha256'])

Vectores verificados: (68973, 384)
Hash: 450d6c1b0274ecda9ef974f33db4504843727942418cc755cd64731ba023695d


## 4. Respaldo, eliminación e inserción

Esta celda modifica ChromaDB. Primero exporta y verifica la colección anterior y después la elimina por API. Respalda también un destino incompatible antes de sustituirlo. Cargas compatibles se reanudan mediante upsert; otras colecciones se conservan. No hay transacción entre SQLite y Chroma: el estado persistido identifica las cargas incompletas.

Para restaurar un respaldo en un nombre libre: `ix.restore(client, ruta_respaldo, target="catalogo_restaurado")`.

In [4]:
collection, state = ix.replace_and_load(client, records, vectors, manifest, run, SALIDAS,
                                         old=COLECCION_ANTERIOR, target=COLECCION)
print('Documentos en Chroma:', collection.count())
print('Respaldos:', state['backups'])

Documentos en Chroma: 68973
Respaldos: ['/home/ariel/Documentos/TesisAgenteCVE/procesoExtraccion/chroma/salidas/respaldos/nvd_cve_2023_2026_20261006T043718956338Z']


## 5. Validación exhaustiva

Se compara cada documento, metadato y vector con la preparación local. Se verifican los 1.000 CVE, las cuotas anuales, la distancia coseno, el límite de tokens y la integridad e invariancia de SQLite.

In [5]:
report = ix.validate(client, collection, records, vectors, manifest, run, DB, tokenizer,
                     old=COLECCION_ANTERIOR)
print(json.dumps({k: report[k] for k in ['collection', 'document_count', 'cve_count', 'years',
      'max_document_tokens', 'integrity_check', 'backups']}, indent=2))
print('Informe:', SALIDAS / 'informe_indexacion.json')

{
  "collection": "catalogo_cve_2023_2026",
  "document_count": 68973,
  "cve_count": 1000,
  "years": {
    "2026": 250,
    "2023": 250,
    "2024": 250,
    "2025": 250
  },
  "max_document_tokens": 240,
  "integrity_check": "ok",
  "backups": [
    "/home/ariel/Documentos/TesisAgenteCVE/procesoExtraccion/chroma/salidas/respaldos/nvd_cve_2023_2026_20261006T043718956338Z"
  ]
}
Informe: /home/ariel/Documentos/TesisAgenteCVE/procesoExtraccion/chroma/salidas/informe_indexacion.json


## 6. Búsqueda y recuperación del catálogo

Las consultas usan el mismo modelo que los documentos. Los resultados se agrupan por CVE antes de recuperar el contexto completo, evitando devolver un CVE repetido por sus fragmentos.

In [6]:
for query in ['Remote code execution caused by command injection',
              'Prevent cross site scripting through output encoding']:
    result = collection.query(query_embeddings=model([query]), n_results=5,
                              include=['documents', 'metadatas', 'distances'])
    ids = list(dict.fromkeys(m['cve_id'] for m in result['metadatas'][0]))
    contexts = ix.retrieve(DB, ids)
    print('Consulta:', query)
    print('CVE:', ids, 'Catálogos recuperados:', len(contexts))
    print('Primer fragmento:', result['documents'][0][0][:500])

Consulta: Remote code execution caused by command injection
CVE: ['CVE-2023-27826', 'CVE-2026-24418', 'CVE-2024-25213', 'CVE-2024-2445', 'CVE-2024-11234'] Catálogos recuperados: 5
Primer fragmento: CVE-2023-27826 | capec | 108 | description
An attacker uses standard SQL injection methods to inject data into the command line for execution. This could be done directly through misuse of directives such as MSSQL_xp_cmdshell or indirectly through injection of data into the database that would be interpreted as shell commands. Sometime later, an unscrupulous backend application (or could be part of the functionality of the same application) fetches the injected data stored in the database and us
Consulta: Prevent cross site scripting through output encoding
CVE: ['CVE-2021-38131', 'CVE-2021-32853', 'CVE-2019-25454', 'CVE-2020-22152', 'CVE-2018-25085'] Catálogos recuperados: 5
Primer fragmento: CVE-2021-38131 | cwe | CWE-79 | potential_mitigations
:Output Encoding:DESCRIPTION:Use and specify 